# CayleyPy Megaminx — quick tour of all 5 shared models

This notebook loads all five trained checkpoints from the [CayleyPy Megaminx Solvers](https://www.kaggle.com/models/artgor/cayleypy-megaminx-solvers) model collection, runs sanity checks on each, and illustrates the **V(V₀) ≈ 2 boundary bug** in standard Bellman training together with the **anchor-mixin fix** that drops V(V₀) to ≈ 0.009.

## What's in the collection

| slug | what it is |
|---|---|
| `m05` | Bellman warm-start V model (production baseline; mean 89.4 strat-5) |
| `m-curr-v3` | + curriculum + frontier replay + BFS-d6 anchor (loss 0.0721) |
| `m-dd-v0` | + V(V₀) anchor-mixin fix (calibrates V at small depths) |
| `m-az-v4` | dual-head value + policy (first 6M model below mean 89 standalone) |
| `m23-v2` | sym-aware Q-shortlister (24-channel; recall 100% at α=2) |

All checkpoints share a `ResMLPDistance` trunk (embedding input, hidden (2048, 512) × 2 ResBlocks, ~6M parameters); the Q-shortlister is wider (12M, output_dim=24); AZ v4 dual-head adds a 24-way policy head and a scalar log_Z parameter.

Each model instance is **self-contained**: bundled `model.py`, `puzzle.py`, `puzzle_info.json`, config, README, and an `inference_example.py`. No need for the cayley repo to load and use any checkpoint.

Runtime: this notebook runs on CPU in ~3 minutes.


## Step 1. Locate the mounted model paths

Each model instance is mounted at `/kaggle/input/cayleypy-megaminx-solvers/PyTorch/<slug>/<version>/`. We grab the latest version of each.


In [ ]:
import os, sys, json, glob
import torch
import numpy as np

# Walk all of /kaggle/input/ and find each variation's mount path by looking
# for the variation slug as a directory name. Kaggle's mount layout varies
# across CLI versions, so we discover at runtime rather than hard-coding.
INPUT_ROOT = '/kaggle/input'
print('full /kaggle/input tree (depth <= 5):')
for path, dirs, files in os.walk(INPUT_ROOT):
    depth = path.replace(INPUT_ROOT, '').count(os.sep)
    indent = '  ' * depth
    print(f'{indent}{os.path.basename(path) or path}/')
    for f in sorted(files):
        print(f'{indent}  {f}')
    if depth >= 5:
        dirs[:] = []

def find_instance(slug: str) -> str:
    """Locate a variation's mounted directory by walking all of /kaggle/input/"""
    aliases = [slug, slug.replace('-', '_')]
    hits = []
    for path, dirs, files in os.walk(INPUT_ROOT):
        if os.path.basename(path) in aliases:
            numeric_children = sorted([d for d in dirs if d.isdigit()], key=int)
            if numeric_children:
                hits.append(os.path.join(path, numeric_children[-1]))
            elif files:
                hits.append(path)
    if not hits:
        # Fallback: look for any directory containing puzzle_info.json alongside this slug
        for path, dirs, files in os.walk(INPUT_ROOT):
            if 'puzzle_info.json' in files and slug in path:
                hits.append(path)
    if not hits:
        raise RuntimeError(f'Could not locate variation {slug!r} under {INPUT_ROOT}')
    return hits[-1]

INSTANCES = {slug: find_instance(slug) for slug in ['m05', 'm-curr-v3', 'm-dd-v0', 'm-az-v4', 'm23-v2']}
print()
print('resolved instance paths:')
for slug, path in INSTANCES.items():
    print(f'  {slug:>12s} -> {path}')


## Step 2. Import the bundled architecture code

Every instance ships an identical `model.py` (containing `ResMLPDistance`) and `puzzle.py` (the `Megaminx` puzzle class). The AZ v4 instance additionally ships `gflow_model.py` for the dual-head model (`ResMLPGFlowNet`).

We put the m05 instance directory on `sys.path` for `model.py` + `puzzle.py`, and the AZ v4 directory for `gflow_model.py`. Either could supply `model.py`; they are byte-identical.


In [ ]:
sys.path.insert(0, INSTANCES['m05'])
from model import ResMLPDistance
from puzzle import Megaminx

sys.path.insert(0, INSTANCES['m-az-v4'])
from gflow_model import ResMLPGFlowNet

# The puzzle definition (24 generators, solved state) is shipped alongside every model.
puzzle = Megaminx.load(os.path.join(INSTANCES['m05'], 'puzzle_info.json'))
print(f'puzzle: {len(puzzle.solved_state)} stickers, {len(puzzle.move_names)} generators')
print(f'generators: {puzzle.move_names}')


## Step 3. Helper loaders

Each checkpoint stores its `model_config` inline. The loader strips the `_orig_mod.` prefix that `torch.compile` adds during training and rebuilds the architecture using the saved config.


In [ ]:
def load_v_model(instance_slug: str, filename: str, device: str = 'cpu') -> ResMLPDistance:
    path = os.path.join(INSTANCES[instance_slug], filename)
    ckpt = torch.load(path, map_location=device, weights_only=False)
    cfg = ckpt['model_config']
    model = ResMLPDistance(**cfg, inference_chunk_size=2048)
    sd = ckpt['state_dict']
    if any(k.startswith('_orig_mod.') for k in sd):
        sd = {k.removeprefix('_orig_mod.'): v for k, v in sd.items()}
    model.load_state_dict(sd)
    model.to(device).eval()
    return model

def load_dual_head(instance_slug: str, filename: str, device: str = 'cpu') -> ResMLPGFlowNet:
    path = os.path.join(INSTANCES[instance_slug], filename)
    ckpt = torch.load(path, map_location=device, weights_only=False)
    cfg = ckpt['model_config']
    model = ResMLPGFlowNet(**cfg, inference_chunk_size=2048)
    sd = ckpt['state_dict']
    if any(k.startswith('_orig_mod.') for k in sd):
        sd = {k.removeprefix('_orig_mod.'): v for k, v in sd.items()}
    model.load_state_dict(sd)
    model.to(device).eval()
    return model


## Step 4. Load all 5 models and inspect V(V₀)

**The bug**: under standard Bellman training, V(V₀) drifts to ≈ 2 instead of the correct boundary value of 0. The cause is that the Bellman target evaluated at the solved state is

$$T(V_0) = 1 + \min_{a} V(V_0 \cdot a) \approx 1 + 1 = 2,$$

because every child of V₀ is at true distance 1 (only V₀ itself is solved). The boundary V(V₀)=0 receives essentially no training signal — V₀ appears once per ~19.3 M states in the BFS-d6 mixin.

**The fix** (m_dd_v0): add 32 copies of V₀ and 4 copies of each of the 24 depth-1 children to every batch with exact targets. Two integer hyperparameters; no other change.

Below: the same V(V₀) measurement on each of our 4 V-only models. m05 and m_curr_v3 sit at the bug's fixed point; m_dd_v0 (anchor-mixin fix) and the AZ v4 V-head (which uses the same anchor recipe via warm-start) are calibrated near 0.


In [ ]:
solved_t = torch.tensor([list(puzzle.solved_state)], dtype=torch.long)

v_models = {}
for slug, fname in [
    ('m05',       'epoch_0499.pt'),
    ('m-curr-v3', 'epoch_0499.pt'),
    ('m-dd-v0',   'epoch_0049.pt'),
    ('m-az-v4',   'v_only.pt'),
]:
    m = load_v_model(slug, fname)
    v_models[slug] = m
    n_params = m.num_parameters()
    v_solved = float(m(solved_t).item())
    marker = '  <-- BUG' if v_solved > 1.0 else '  <-- fixed'
    print(f'{slug:>12s}  params={n_params:>10,}  V(solved) = {v_solved:.4f}{marker}')


## Step 5. Same models on a scrambled state

All four V-only models predict similar (large) values for the same scrambled state. The boundary fix doesn't change the bulk of the heuristic — it only corrects the goal-state calibration.


In [ ]:
import random
rng = random.Random(0)
scrambled = list(puzzle.solved_state)
for _ in range(40):
    scrambled = list(puzzle.apply_move(scrambled, rng.choice(puzzle.move_names)))
scrambled_t = torch.tensor([scrambled], dtype=torch.long)

print('V at a 40-move scramble:')
for slug, m in v_models.items():
    v_scrambled = float(m(scrambled_t).item())
    print(f'  {slug:>12s}  V(scrambled) = {v_scrambled:.3f}')


## Step 6. AZ v4 dual-head: value + policy in one model

The dual-head model exposes both a value head (which we saw above as the extracted `v_only.pt`) and a 24-way policy head. The policy head was trained by cross-entropy on solver paths from the merged 76,304-move competition submission.

Note on the value convention: the dual-head model reports the value as the GFlowNet log-flow (`log F(s) ≈ const - dist(s, V_0)`). The extracted V-only checkpoint flips the sign so it reads as a positive distance estimate; if you use the dual-head directly, remember to negate `value` for beam-search ranking.


In [ ]:
az_v4 = load_dual_head('m-az-v4', 'epoch_0024_dual_head.pt')
print(f'AZ v4 dual-head: {az_v4.num_parameters():,} parameters')
with torch.no_grad():
    pi_logits_s, value_s = az_v4(solved_t)
    pi_logits_x, value_x = az_v4(scrambled_t)

print(f'  value(solved)    = {float(value_s.item()):.4f}     (log-flow; near 0)')
print(f'  value(scrambled) = {float(value_x.item()):.4f}     (log-flow; should be negative)')

probs = torch.softmax(pi_logits_x[0], dim=-1)
top5 = torch.topk(probs, 5)
print()
print('Top-5 policy actions on the scrambled state:')
for p, idx in zip(top5.values.tolist(), top5.indices.tolist()):
    print(f'  {puzzle.move_names[idx]:>4s}   prob = {p:.3f}')


## Step 7. m23_v2 — the sym-aware Q-shortlister

m23_v2 is a 24-output Q model: for each parent state s, it predicts a vector of 24 scalars Q_a(s) ≈ V_m05(s · a) — one per generator. In production beam search at width B, m23_v2 runs once per parent (B forwards total) and supplies a shortlist of αB candidates that the full V teacher then reranks. At α=2 the recall against the m05 top-B is 100%, with a ~4.4× wall reduction at B=131k.

The "sym-aware" variant was distilled with 50% per-batch rotation augmentation using the megaminx symmetry group $A_5 \times C_6$ (360 elements). This is what lets it compose with the `--sym-ensemble` inference mechanism.


In [ ]:
qshort = load_v_model('m23-v2', 'epoch_0499.pt')  # output_dim=24
print(f'm23_v2: {qshort.num_parameters():,} parameters, output_dim={qshort.output_dim}')

with torch.no_grad():
    q_vec = qshort(scrambled_t)[0]   # (24,) — predicted V of each child

sorted_idx = torch.argsort(q_vec).tolist()
names = list(puzzle.move_names)
print()
print('Top-5 actions by predicted child V (lowest = most promising):')
for rank, idx in enumerate(sorted_idx[:5]):
    print(f'  {rank+1}. {names[idx]:>4s}   Q = {float(q_vec[idx].item()):.3f}')


## Step 8. Summary table

Everything in one frame for easy comparison.


In [ ]:
import pandas as pd

rows = []
for slug, m in v_models.items():
    rows.append({
        'model': slug,
        'params': m.num_parameters(),
        'V(V_0)': round(float(m(solved_t).item()), 4),
        'V(scrambled)': round(float(m(scrambled_t).item()), 3),
    })
df = pd.DataFrame(rows)
df


## What to do next

- For a worked submission baseline using AZ v4 V-only on the competition test set, see the companion notebook **`megaminx-baseline-with-az-v4`** in the same user account.
- For full reproducibility, see each model instance's `README.md` and `config.yaml` (the training recipe is fully specified). The architecture source is bundled in `model.py` / `gflow_model.py` so the checkpoints load standalone.
- For the **megaminx symmetry group `A₅ × C₆`** mechanism and the `--sym-ensemble K=4` production trick that pairs with m23_v2, see the corresponding section in our paper. The 360-element rotation table is small (~42 KB int8) and can be regenerated from the symmetry-derivation script in our project repository.

If you find these checkpoints useful in your own work, please cite the CayleyPy Megaminx paper. The competition itself is open: <https://www.kaggle.com/competitions/cayley-py-megaminx>.
